# Transformation 02 — Feature Pruning

Drop columns that are:
- **High-cardinality noise**: `DBA Name`, `AKA Name`, `Address` — free-text fields
  that are nearly unique per row; unlikely to generalize.
- **Constant / near-constant flags**: `flag_non_il_state`, `flag_longitude_outside_typical_range`.
- **Raw date columns**: after feature extraction, the original datetime columns are no longer needed.

In [43]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t01.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t01.parquet'))
print('Train shape BEFORE pruning:', train.shape)
print('Test  shape BEFORE pruning:', test.shape)
print()
print('Columns:', list(train.columns))

Train shape BEFORE pruning: (137176, 28)
Test  shape BEFORE pruning: (34294, 28)

Columns: ['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE', 'LICENSE STATUS', 'flag_non_il_state', 'flag_non_chicago_city', 'flag_longitude_outside_typical_range', 'violations_recorded', 'license_matched', 'has_prior_inspection', 'days_to_license_expiry', 'license_expiry_missing']


## 1 · Identify columns to drop

In [44]:
# High-cardinality noise columns
NOISE_COLS = ['DBA Name', 'AKA Name', 'Address']

# Constant / near-constant flag columns
FLAG_COLS = ['flag_non_il_state', 'flag_longitude_outside_typical_range', 'flag_non_chicago_city']

# Raw date columns (features have been extracted in previous step)
RAW_DATE_COLS = ['LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE']

# Combine all columns to drop
DROP_COLS = []
for col in NOISE_COLS + FLAG_COLS + RAW_DATE_COLS:
    if col in train.columns:
        DROP_COLS.append(col)
    else:
        print(f'Column "{col}" not found — skipping.')

print(f'Columns to drop ({len(DROP_COLS)}):', DROP_COLS)

Columns to drop (8): ['DBA Name', 'AKA Name', 'Address', 'flag_non_il_state', 'flag_longitude_outside_typical_range', 'flag_non_chicago_city', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE']


## 2 · Quick check on constant flags

In [45]:
for col in FLAG_COLS:
    if col in train.columns:
        vc = train[col].value_counts()
        print(f'{col}:')
        print(vc)
        print(f'  -> unique values: {train[col].nunique()}')
        print()

flag_non_il_state:
flag_non_il_state
False    137176
Name: count, dtype: int64
  -> unique values: 1

flag_longitude_outside_typical_range:
flag_longitude_outside_typical_range
False    137176
Name: count, dtype: int64
  -> unique values: 1

flag_non_chicago_city:
flag_non_chicago_city
False    137027
True        149
Name: count, dtype: int64
  -> unique values: 2



## 3 · Drop columns

In [46]:
train = train.drop(columns=DROP_COLS)
test  = test.drop(columns=DROP_COLS)

print('Train shape AFTER pruning:', train.shape)
print('Test  shape AFTER pruning:', test.shape)
print()
print('Remaining columns:', list(train.columns))

Train shape AFTER pruning: (137176, 20)
Test  shape AFTER pruning: (34294, 20)

Remaining columns: ['Inspection ID', 'License #', 'Facility Type', 'Risk', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'COMMUNITY AREA NAME', 'LICENSE DESCRIPTION', 'APPLICATION TYPE', 'LICENSE STATUS', 'violations_recorded', 'license_matched', 'has_prior_inspection', 'days_to_license_expiry', 'license_expiry_missing']


In [47]:
print(train['Facility Type'].nunique())
print(train['Facility Type'].value_counts().head(10))

431
Facility Type
Restaurant                         92150
Grocery Store                      18606
School                              9386
Bakery                              2035
Daycare (2 - 6 Years)               1992
Children's Services Facility        1912
Daycare Above and Under 2 Years     1648
Long Term Care                       898
Catering                             786
Mobile Food Dispenser                725
Name: count, dtype: int64


## 4 · Save intermediate results

In [48]:
train.to_parquet(DataLoader.transformed('train_t02.parquet'), index=False)
test.to_parquet(DataLoader.transformed('test_t02.parquet'), index=False)
print('Saved train_t02.parquet and test_t02.parquet')

Saved train_t02.parquet and test_t02.parquet


In [49]:
train.columns.tolist()

['Inspection ID',
 'License #',
 'Facility Type',
 'Risk',
 'Zip',
 'Inspection Date',
 'Inspection Type',
 'Results',
 'Violations',
 'Latitude',
 'Longitude',
 'COMMUNITY AREA NAME',
 'LICENSE DESCRIPTION',
 'APPLICATION TYPE',
 'LICENSE STATUS',
 'violations_recorded',
 'license_matched',
 'has_prior_inspection',
 'days_to_license_expiry',
 'license_expiry_missing']